## Step 1: Converting the Date Column to Proper Date Type

* `parse_dates=['col_name']` → Passing this inside `read_csv()` automatically converts the column to datetime format
* Replaces the need to call `pd.to_datetime()` separately afterwards as a shortcut

In [ ]:
import pandas as pd
df = pd.read_csv("../datasets/weather_data.csv", parse_dates=['day'])
df

## Step 2: Setting Date as Index

* `set_index()` → Same previous concept; now that the date is in a proper datetime format, time-based operations (like `interpolate(method="time")`) will work correctly

In [ ]:
df.set_index('day',inplace=True)
df

## Handling Missing Values using fillna()

* `df.fillna(0)` → Replaces all missing (`NaN`) values across the entire DataFrame with `0`
* Dictionary Mapping → Replaces missing (`NaN`) values with column-specific defaults (e.g., numeric defaults for measurements and string defaults for categorical data)

In [ ]:
new_df = df.fillna(0)
new_df

In [ ]:
new_df = df.fillna({
    'temperature' : 0,
    'windspeed' : 0,
    'event' : "no event"
})
new_df

## ffill() and bfill() — Filling Values from Adjacent Cells

* `ffill()` → Fill `NaN` with the previous (above) value (Forward Fill)
* `bfill()` → Fill `NaN` with the next (below) value (Backward Fill)
* `axis="columns"` → Fill horizontally from LEFT-to-RIGHT across columns within a row (less common use case)
*(Default is `axis="index"`, which fills vertically/top-to-bottom across rows)*

In [ ]:
new_df = df.ffill()
new_df

In [ ]:
new_df = df.bfill()
new_df

In [ ]:
new_df = df.bfill(axis="columns")
new_df

In [ ]:
new_df = df.ffill(axis="columns")
new_df

## limit Parameter — Restricting Fill Frequency

* `ffill(limit=1)` → Fills only 1 consecutive `NaN` value; subsequent `NaN`s remain unchanged
* Real-world use case: Ideal for filling small gaps (1-2 missing days), whereas filling large gaps (a week of missing data) would be misleading

In [ ]:
new_df = df.ffill(limit=1)
new_df


## interpolate() — Filling Values Using Mathematical Estimation

* `method="time"` → Accurately estimates values based on actual date-index intervals *(requires the index to be a `DatetimeIndex`)*
* IMPORTANT: Calling `interpolate()` on text (`object` dtype) columns will trigger a warning
* FIX: Filter numeric columns first using `select_dtypes(include='number')`, then apply `interpolate()`
*(The `numeric_only=True` parameter is unreliable in this version — `select_dtypes` is the preferred approach)*

In [ ]:
new_df = df.select_dtypes(include='number').interpolate(method="time")
new_df

## dropna() — Dropping Rows with Missing Data

* `dropna()` → Drops the entire row if even a SINGLE `NaN` is present (most aggressive/strict)
* `dropna(how='all')` → Drops the row only if ALL values are `NaN` (least aggressive)
* `dropna(thresh=n)` → Requires at least `n` non-NaN values for the row to be kept, otherwise drops it (balanced approach)

In [ ]:
new_df = df.dropna()
new_df

In [ ]:
new_df = df.dropna(how='all')
new_df

In [ ]:
new_df = df.dropna(thresh=2)
new_df

## reindex() — Explicitly Adding Missing Dates

* `pd.date_range(start, end)` → Generates a COMPLETE sequence of dates between the start and end dates (no skipped days)
* `reindex(idx)` → Forces the DataFrame to conform to this complete date sequence
* Dates absent in the original data are automatically populated with `NaN` across columns
* Real-world use case: Reveals missing time-series dates explicitly as `NaN` instead of leaving gaps undetected, allowing for proper data handling

In [ ]:
dt = pd.date_range("2017-01-01", "2017-01-11")
idx = pd.DatetimeIndex(dt)

df = df.reindex(idx)
df